# Imports

In [2]:
import librosa
import pandas as pd
from typing import Callable, Union, List
import os
from pathlib import Path
import sys
from pprint import pprint

from audio_dataset import RavdessRawData
from Preprocess import audio_to_waveform, trim_silence

# Functions

In [ ]:
def extract_audio_statistics(
    audio_paths: List[Path],
    stat_func: Callable[[Path], dict]
) -> pd.DataFrame:
    """
    Generate a DataFrame with statistics for each audio file.

    Each row corresponds to an audio file.
    Columns include the file path and attributes returned by `stat_func`.

    Parameters:
    - audio_paths: List of audio file paths.
    - stat_func: A function that takes a path and returns a dict of stats.

    Returns:
    - pd.DataFrame with one row per file and one column per attribute.
    """
    records = []
    for path in audio_paths:
        stats = stat_func(path)
        stats["path"] = str(path)
        records.append(stats)
    return pd.DataFrame(records)


def no_silence_duration_stat(path: Path) -> dict:
    """
    function to extract duration of the no-silence part of an audio file. meaning the duration after trimming silence from the start and end of the audio file.
    """
    # Here you would implement the logic to get the duration of the audio file.
    waveform, sample_rate = audio_to_waveform(path)
    trimmed_waveform = trim_silence(waveform)
    duration = librosa.get_duration(y=trimmed_waveform, sr=sample_rate)
    return {"duration": duration} 

# Change Working Dir To the Project Working Dir

In [3]:
# change the dir to the grandparent directory of the current working directory

current_dir = Path(os.getcwd())
grandparent_dir = current_dir.parent.parent
os.chdir(grandparent_dir)

In [4]:
os.getcwd()  # Check the cwd has updated

'c:\\Users\\noams\\Python Projects\\Audio_processing_project'

# Data Examination

In [18]:
ravdess_raw_data = RavdessRawData()
audio_paths_with_labels = list(ravdess_raw_data.all_data)
audio_paths = [path for path, _ in audio_paths_with_labels]
ravdess_silenced_duraion = extract_audio_statistics(audio_paths, no_silence_duration_stat)
print(ravdess_silenced_duraion.head())  # Display the first few rows of the DataFrame

   duration                                               path
0     1.664  RAVDESS\original_data\Actor_18\03-01-03-02-02-...
1     1.952  RAVDESS\original_data\Actor_12\03-01-03-02-02-...
2     1.888  RAVDESS\original_data\Actor_08\03-01-07-01-02-...
3     1.344  RAVDESS\original_data\Actor_13\03-01-08-02-01-...
4     2.560  RAVDESS\original_data\Actor_03\03-01-06-02-01-...


In [21]:
# show statistics of the audio files
ravdess_silenced_duraion.describe()  # Display the statistics of the DataFrame

,duration
count,1440.000000
mean,1.732832
std,0.349538
min,0.864000
25%,1.504000
50%,1.664000
75%,1.920000
max,3.412437


In [5]:
from IPython.display import display

from pathlib import Path
# from captum.attr import LayerActivation
# from functorch.dim import Tensor #! makes a bug because functorch.dim isn't supported in python 3.12 !!
from pprint import pprint
from typing import Optional

import numpy as np
import pandas as pd
import torch
from captum.concept import TCAV, Concept
from torch.utils.data import DataLoader, Dataset

from Preprocess import audio_to_mel_spectrogram
from PreprocessParams import TARGET_FRAMES, FREQUENCY_BIN_COUNT
from concepts_creation import generate_random_pattern_spectrogram


class PreGeneratedRandomSpectrogramDataset(Dataset):
    """
    PyTorch Dataset that pre-generates all random spectrogram in memory.
    """

    def __init__(self, n_samples: int, freq_count = FREQUENCY_BIN_COUNT, frames = TARGET_FRAMES, rng_seed: Optional[int] = None):
        self.n_samples = n_samples
        self.freq_count = freq_count
        self.frames = frames
        self.rng = np.random.default_rng(rng_seed)

        # Pre-generate all spectrograms in memory
        self.data = np.array([generate_random_pattern_spectrogram(freq_count, frames, rng=self.rng)
                     for _ in range(n_samples)])
        self.data = torch.tensor(self.data, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Ensure shape [1, H, W] per sample
        x = self.data[idx]
        return x.unsqueeze(0)

    @property
    def get_data(self):
        return self.data
    
class PreGeneratedConceptDataset(Dataset):
    """
    PyTorch Dataset that pre-generates the dataset for a specific concept in memory.
    """

    def __init__(self, n_samples: int, concept_name: str, root_concept_dir: Path = Path("positive concepts dataset") , freq_count = FREQUENCY_BIN_COUNT, frames_count = TARGET_FRAMES, rng_seed: Optional[int] = None):
        self.n_samples = n_samples
        self.concept_name = concept_name
        self.root_concept_dir = root_concept_dir
        self.freq_count = freq_count
        self.frames = frames_count
        self.rng = np.random.default_rng(rng_seed)

        # load all .npy files from root_concept_dir/concept_name
        self.data = []
        concept_dir = self.root_concept_dir / self.concept_name
        concept_dir.mkdir(exist_ok=True)
        for npy_file in concept_dir.glob("*.npy"):
            self.data.append(np.load(npy_file))
        self.data = np.array(self.data)
        self.data = torch.tensor(self.data, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Ensure shape [1, H, W] per sample
        x = self.data[idx]
        return x.unsqueeze(0)

    @property
    def get_data(self):
        return self.data

concept_unique_names = [
                        "long_constant_thick",
                        "long_dropping_flat_thick",
                        "long_dropping_steep_thick",
                        "long_dropping_steep_thin",
                        "long_rising_flat_thick",
                        "long_rising_steep_thick",
                        "long_rising_steep_thin",
                        "short_constant_thick",
                        "short_dropping_steep_thick",
                        "short_dropping_steep_thin",
                        "short_rising_steep_thick",
                        "short_rising_steep_thin"
                        ]

index_emotion_mapping = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised',
}

label_emotion_mapping = {
    0: 'angry', 1: 'calm', 2: 'disgust', 3: 'fearful',
    4: 'happy', 5: 'neutral', 6: 'sad', 7: 'surprised',
}

def get_emotion_tensor(emotion_label: str, drop_false_positive: bool) -> torch.Tensor:
    """
    return emotion tensor containing all the spectrograms that the model predicted as "emotion_label"

    Args:
        emotion_label (str): The emotion label to filter by.
        drop_false_positive (bool): e.g. if emotion_label='angry', then audio classified as 'angry' but not actually 'angry' will be dropped.

    Returns:
        torch.Tensor: A tensor containing the spectrograms for the specified emotion label. in shape: [Batch, 1, Height, Width]
    """
    df = pd.read_csv("attributes/all_attributes.csv")

    df_emotion = df[(df["predicted_label"] == emotion_label) & (df["true_label"] == emotion_label)]["path"] if drop_false_positive else df[(df["predicted_label"] == emotion_label)]["path"]
    
    # audio_to_mel_spectrogram to all audio samples
    mel_specs = [audio_to_mel_spectrogram(Path(path)) for path in df_emotion]

    # stack all the mel spectrograms to one big tensor
    emotion_tensor = torch.stack([torch.tensor(spec) for spec in mel_specs])
    
    # if shape=[B,H,W] change it to [B,1,H,W]
    if emotion_tensor.dim() == 3:
        emotion_tensor = emotion_tensor.unsqueeze(1)  # [B, 1, H, W]
        
    return emotion_tensor

def tcav_scores_to_df(scores_by_label: dict, concept_names: list[str]) -> pd.DataFrame:
    """
    Flatten Captum TCAV results into a DataFrame with:
    columns = ["label_name", "concept_name", "layer_name", "positive_sign_count", "positive_magnitude"]
    """
    rows = []
    for label_name, exp_sets in scores_by_label.items():
        # exp_key looks like "0-12" where 0 is the positive concept index, 12 is random/baseline
        for exp_key, layer_dict in exp_sets.items():
            try:
                pos_idx = int(str(exp_key).split("-")[0])
            except Exception:
                continue  # skip malformed keys
            if not (0 <= pos_idx < len(concept_names)):
                continue
            concept_name = concept_names[pos_idx]

            # Usually there's a single chosen layer, but handle multiple layers just in case
            for layer_name, metrics in layer_dict.items():
                sc = metrics.get("sign_count")
                mg = metrics.get("magnitude")
                if sc is None or mg is None:
                    continue

                # Convert torch tensors to Python floats
                if isinstance(sc, torch.Tensor):
                    sc = sc.detach().cpu().tolist()
                if isinstance(mg, torch.Tensor):
                    mg = mg.detach().cpu().tolist()

                # Positive direction = index 0
                rows.append({
                    "label_name": label_name,
                    "concept_name": concept_name,
                    "layer_name": layer_name,
                    "positive_sign_count": float(sc[0]),
                    "positive_magnitude": float(mg[0]),
                })

    return pd.DataFrame(rows, columns=[
        "label_name", "concept_name", "layer_name", "positive_sign_count", "positive_magnitude"
    ])

# store spectrograms of each emotion in Tensor object.
label_inputs = {
    label_name: get_emotion_tensor(label_name, drop_false_positive=True)
    for label_name in label_emotion_mapping.values()
}


# -----------------------------
# 1️⃣ Load pretrained model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.load(Path("ResNetWithAttention.pt"), map_location=device, weights_only=False)
model.eval()

# -----------------------------
# 2️⃣ Choose layer for TCAV to work on
# -----------------------------

layer = "module3.blocks.0.conv2"

# -----------------------------
# 3️⃣ Compute TCAV
# -----------------------------
# Captum TCAV expects a dictionary of concept activations, with positive and negative examples.


# Define TCAV object
tcav = TCAV(model, [layer])
tcav_scores_per_label= {}


positive_concepts: list[Concept] = [Concept(id=concept_idx, name=concept_name, data_iter=DataLoader(PreGeneratedConceptDataset(n_samples=60, concept_name=concept_name), shuffle=False))
                               for concept_idx, concept_name in enumerate(concept_unique_names)]

# This concept is the negative of concepts.
negative_concept_dataset = PreGeneratedRandomSpectrogramDataset(n_samples=10, freq_count=FREQUENCY_BIN_COUNT, frames=TARGET_FRAMES)
random_concept = Concept(id=len(positive_concepts), name='random', data_iter=DataLoader(negative_concept_dataset, shuffle=False))

# Debug call, don't uncomment
# show_arrays_in_separate_windows(negative_concept_dataset.get_data)

print("Reached tcav interpret")
for label_index, label_name in label_emotion_mapping.items():
    tcav_scores_per_label[label_name] = tcav.interpret(
        inputs=label_inputs[label_name],
        experimental_sets=[[c, random_concept] for c in positive_concepts],
        target=label_index  # integer index of target class
    )

# -----------------------------
# 4️⃣ Inspect results
# -----------------------------

pprint(tcav_scores_per_label)

# pprint(f"TCAV scores for label {'angry'}: {tcav_scores_per_label['angry']}", depth=1)

df_tcav = tcav_scores_to_df(tcav_scores_per_label, concept_unique_names)

display(df_tcav)

c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\captum\concept\_utils\classifier.py:130: UserWarning: Using default classifier for TCAV which keeps input both train and test datasets in the memory. Consider defining your own classifier that doesn't rely heavily on memory, for large number of concepts, by extending `Classifer` abstract class
  warnings.warn(


Reached tcav interpret


c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\captum\concept\_core\cav.py:165: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  save_dict = torch.load(cavs_path)


{'angry': defaultdict(<function TCAV.interpret.<locals>.<lambda> at 0x000001C698B3B2E0>,
                      {'0-12': defaultdict(None,
                                           {'module3.blocks.0.conv2': {'magnitude': tensor([ 0.3043, -0.3043]),
                                                                       'sign_count': tensor([0.7594, 0.2406])}}),
                       '1-12': defaultdict(None,
                                           {'module3.blocks.0.conv2': {'magnitude': tensor([ 1.7341, -1.7341]),
                                                                       'sign_count': tensor([0.9947, 0.0053])}}),
                       '10-12': defaultdict(None,
                                            {'module3.blocks.0.conv2': {'magnitude': tensor([-0.1385,  0.1385]),
                                                                        'sign_count': tensor([0.3636, 0.6364])}}),
                       '11-12': defaultdict(None,
                                 

,label_name,concept_name,layer_name,positive_sign_count,positive_magnitude
0,angry,long_constant_thick,module3.blocks.0.conv2,0.759358,0.304350
1,angry,long_dropping_flat_thick,module3.blocks.0.conv2,0.994652,1.734131
2,angry,long_dropping_steep_thick,module3.blocks.0.conv2,0.080214,-0.517384
3,angry,long_dropping_steep_thin,module3.blocks.0.conv2,0.994652,1.329065
4,angry,long_rising_flat_thick,module3.blocks.0.conv2,0.000000,-0.912052
...,...,...,...,...,...
91,surprised,short_constant_thick,module3.blocks.0.conv2,0.920904,0.587009
92,surprised,short_dropping_steep_thick,module3.blocks.0.conv2,0.937853,0.601205
93,surprised,short_dropping_steep_thin,module3.blocks.0.conv2,0.887006,0.618544
94,surprised,short_rising_steep_thick,module3.blocks.0.conv2,0.994350,1.807984


In [16]:
df_2_show = df_tcav[(df_tcav["label_name"] == "calm")].drop("layer_name", axis=1)

display(df_2_show)
print(df_2_show.shape)

,label_name,concept_name,positive_sign_count,positive_magnitude
12,calm,long_constant_thick,0.234973,-0.390145
13,calm,long_dropping_flat_thick,0.005464,-1.826928
14,calm,long_dropping_steep_thick,0.267760,-0.345932
15,calm,long_dropping_steep_thin,0.103825,-1.042938
16,calm,long_rising_flat_thick,0.295082,-0.144539
17,calm,long_rising_steep_thick,0.049180,-1.051690
18,calm,long_rising_steep_thin,0.169399,-0.490444
19,calm,short_constant_thick,0.338798,-0.202826
20,calm,short_dropping_steep_thick,0.245902,-0.261732
21,calm,short_dropping_steep_thin,0.060109,-0.843508


(12, 4)
